In [125]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MultiLabelBinarizer

from pathlib import Path
import warnings

In [126]:
warnings.filterwarnings("ignore")

ACCENT_PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff', '#f85149', '#56d4dd']

pd.set_option({
    "display.max_rows": None,
    "display.max_columns": None,
    "display.width": None,
    "display.max_colwidth": None,
})

plt.rcParams.update({
    'figure.facecolor': '#0d1117',

    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'axes.titlecolor': '#e6edf3',
    'axes.prop_cycle': plt.cycler(color=ACCENT_PALETTE),

    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',

    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#cccccc',
    'grid.linestyle': "--",
    'grid.alpha': 0.75,

    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',

    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

sns.set_palette(ACCENT_PALETTE)

In [127]:
def load_data(path: str) -> pd.DataFrame:
    data_path = Path(path)

    if not data_path.exists():
        raise FileNotFoundError(f'{data_path} does not exist')

    return pd.read_csv(data_path)

df = load_data("../data/cleaned-data/cleaned-data.csv")

In [128]:
print("==========INFO==========\n")
print(f"Df shape: {df.shape}\n")
print(f"Df info: {df.info()}\n")
print(f"Df describe: {df.describe()}\n")

==========INFO==========

Df shape: (161, 50)

<class 'pandas.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 50 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    161 non-null    int64  
 1   title                 161 non-null    str    
 2   year                  161 non-null    int64  
 3   type                  161 non-null    str    
 4   rated                 161 non-null    str    
 5   runtime_min           161 non-null    float64
 6   genre                 161 non-null    str    
 7   director              161 non-null    str    
 8   writer                161 non-null    str    
 9   actors                161 non-null    str    
 10  plot                  161 non-null    str    
 11  language              161 non-null    str    
 12  country               161 non-null    str    
 13  awards                161 non-null    str    
 14  imdb_rating           161 non-null    

In [129]:
similar = {
    "imdb_rating": "IMDb",
    "rt_score": "Rotten Tomatoes",
    "metacritic_score": "Metacritic",
    "tmdb_rating": "TMDB",
}

for master_col, ratings_col in similar.items():
    diff = (df[master_col] - df[ratings_col]).abs()
    print(f"{master_col} vs {ratings_col}:")
    print(diff.describe())
    print(f"Max diff: {diff.max()}, rows with diff > 0.5: {(diff > 0.5).sum()}\n")

diff_rt = (df["rt_score"]/10 - df["Rotten Tomatoes"]).abs()
print(diff_rt.describe())

diff_meta = (df["metacritic_score"]/10 - df["Metacritic"]).abs()
print(diff_meta.describe())

df = df.drop(columns=list(similar.keys()))

imdb_rating vs IMDb:
count    161.000000
mean       0.067903
std        0.089560
min        0.000000
25%        0.000000
50%        0.000000
75%        0.185294
max        0.185294
dtype: float64
Max diff: 0.18529411764705817, rows with diff > 0.5: 0

rt_score vs Rotten Tomatoes:
count    161.000000
mean      64.570067
std       17.176376
min        8.100000
25%       69.300000
50%       69.396154
75%       69.396154
max       87.300000
dtype: float64
Max diff: 87.3, rows with diff > 0.5: 161

metacritic_score vs Metacritic:
count    161.000000
mean      56.225647
std        8.863052
min       24.300000
25%       57.995833
50%       57.995833
75%       57.995833
max       79.200000
dtype: float64
Max diff: 79.2, rows with diff > 0.5: 161

tmdb_rating vs TMDB:
count    161.000000
mean       0.092793
std        0.519448
min        0.000000
25%        0.000000
50%        0.000000
75%        0.190573
max        6.554427
dtype: float64
Max diff: 6.554427350427351, rows with diff > 0.5: 1

c

In [130]:
print("==========STRING COLUMNS==========\n")
str_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Str columns: {str_cols}")

df = df.drop(columns=["title", "top_actor"])

categorial_cols = ["type", "rated", "mcu_phase", "universe", "decade", "status", "language", "country"]
for col in categorial_cols:
    print(f"Categorial columns: {col} - {df[col].nunique()} values")

df = df.drop(columns=["language", "country"])

categorial_cols.remove("language")
categorial_cols.remove("country")
df = pd.get_dummies(df, columns=categorial_cols, drop_first=True)

unique_cols = ["director", "writer", "actors", "directors", "producers", "composer", "top5_cast", "tmdb_keywords"]
df = df.drop(columns=unique_cols)

text_cols = ["plot", "awards", "tagline", "document"]
df = df.drop(columns=text_cols)

print("\n==========REMAINING STRING COLUMNS==========")
print(df.select_dtypes(include="object").columns.tolist())

==========STRING COLUMNS==========

Str columns: ['title', 'type', 'rated', 'genre', 'director', 'writer', 'actors', 'plot', 'language', 'country', 'awards', 'mcu_phase', 'universe', 'decade', 'tmdb_genres', 'top5_cast', 'directors', 'producers', 'composer', 'tmdb_keywords', 'production_countries', 'spoken_languages', 'status', 'tagline', 'document', 'top_actor']
Categorial columns: type - 3 values
Categorial columns: rated - 11 values
Categorial columns: mcu_phase - 7 values
Categorial columns: universe - 5 values
Categorial columns: decade - 7 values
Categorial columns: status - 6 values
Categorial columns: language - 45 values
Categorial columns: country - 41 values

==========REMAINING STRING COLUMNS==========
['genre', 'tmdb_genres', 'production_countries', 'spoken_languages']


In [131]:
print(df[["genre", "tmdb_genres"]].head(10))
print(df.loc[[6, 7], ["genre", "tmdb_genres"]])

genre_values = set(g.strip() for row in df["genre"].str.split(",") for g in row)
tmdb_genre_values = set(g.strip() for row in df["tmdb_genres"].str.split(",") for g in row)

print(f"\nOnly in gender: {genre_values - tmdb_genre_values}")
print(f"Only in tmdb_genres: {tmdb_genre_values - genre_values}")
print(f"Main: {genre_values & tmdb_genre_values}")

genre_synonyms = {
    "Science Fiction": "Sci-Fi",
    "Sci-Fi & Fantasy": "Sci-Fi, Fantasy",
    "Action & Adventure": "Action, Adventure",
    "Talk": "Talk-Show",
    "Kids": "Family",
}

def normalize_genres(genre_str, synonyms):
    parts = [g.strip() for g in genre_str.split(",")]
    normalized = []
    for p in parts:
        replacement = synonyms.get(p, p)
        normalized.extend(r.strip() for r in replacement.split(","))
    return list(set(normalized))

df["tmdb_genres_norm"] = df["tmdb_genres"].apply(lambda x: normalize_genres(x, genre_synonyms))
df["genre_norm"] = df["genre"].apply(lambda x: normalize_genres(x, {}))

df["all_genres"] = df.apply(lambda row: list(set(row["genre_norm"]) | set(row["tmdb_genres_norm"])), axis=1)

df["is_tv_movie_format"] = df["tmdb_genres"].str.contains("TV Movie").astype(int)

df = df.drop(columns=["tmdb_genres", "genre", "tmdb_genres_norm", "genre_norm"])

mlb = MultiLabelBinarizer()
genre_encoded = pd.DataFrame(
    mlb.fit_transform(df["all_genres"]),
    columns=[f"genre_{g}" for g in mlb.classes_],
    index=df.index
)

df = pd.concat([df.drop(columns=["all_genres"]), genre_encoded], axis=1)

print(df.select_dtypes(include="object").columns.tolist())

prod_countries_values = set(c.strip() for row in df["production_countries"].str.split(",") for c in row)
spoken_lang_values = set(l.strip() for row in df["spoken_languages"].str.split(",") for l in row)

print("\n\n==========REMAINING COLUMNS==========")
print(f"Production countries ({len(prod_countries_values)}): {prod_countries_values}")
print(f"Spoken languages ({len(spoken_lang_values)}): {spoken_lang_values}")

df = df.drop(columns=["production_countries", "spoken_languages"])

print("\n\n==========DF AFTER ENCONDING==========")
print(df.info())

                       genre  \
0  Action, Adventure, Sci-Fi   
1   Action, Adventure, Crime   
2   Action, Adventure, Crime   
3            Action, Fantasy   
4                    Unknown   
5    Action, Family, Fantasy   
6      Short, Action, Sci-Fi   
7   Action, Adventure, Crime   
8  Action, Adventure, Sci-Fi   
9  Action, Adventure, Sci-Fi   

                                             tmdb_genres  
0                     Action, Adventure, Science Fiction  
1                         Science Fiction, Action, Crime  
2                         Science Fiction, Action, Crime  
3  TV Movie, Action, Adventure, Fantasy, Science Fiction  
4                         Science Fiction, Action, Drama  
5           Action, Adventure, Family, Fantasy, TV Movie  
6           Action, Adventure, Family, Fantasy, TV Movie  
7                                                Unknown  
8           Action, Adventure, Science Fiction, TV Movie  
9               Action, Drama, Science Fiction, TV Movie 